##### NanoMaker use / execution workflow
This notebook will allow you to generate protein binding pockets for any chemical provided in SMILES format. The comments in each of the cells will walk you through.

In [1]:
from paths import *
from nano_maker.utility import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import *

In [2]:
# initialize NanoMaker system

model = NanoMaker(
    version_code=version_code,
    skeleton_weight_filename=skeleton_weight_filename,
    naanobot_weight_filename=naanobot_weight_filename,
    skeleton_cfg=skeleton_config,
    naanobot_cfg=naanobot_config,
    radial_cfg=radial_config)

In [14]:
# pt 1 -> let's create a binding pocket for a given drug
# - generate the binding pocket data
# - then save it to a .nanopkt file format
drug_i_want_to_deliver = "CC1=C2C(C(=C(C3=C(C(=O)C(C3(C1(C(CC2(C(C(C(C(=O)O)C(C(=O)NC(C(C5=CC=CC=C5)C(O)C6=CC=CC=C6)=O)O)C)C)OC(=O)C)(C)O)O)C)O)C)C)OC(=O)C)OC(=O)C"
model.ingest_chemical(drug_i_want_to_deliver)

# you won't always need to save it to a file format but it's handy.

Chemical Ingested: CC1=C2C(C(=C(C3=C(C(=O)C(C3(C1(C(CC2(C(C(C(C(=O)O)C(C(=O)NC(C(C5=CC=CC=C5)C(O)C6=CC=CC=C6)=O)O)C)C)OC(=O)C)(C)O)O)C)O)C)C)OC(=O)C)OC(=O)C
Scaffold: O=C1C=C2C=CCC3=CC(CCC3CCCCC(=O)NC(=O)C(Cc3ccccc3)c3ccccc3)C2C1


In [15]:
pocket_data = model.generate_nanopkt_data(sampling_temp=0.3)
# sampling temperature is on a 0-1 float scale -> determines amino acid sampling strictness
# lower <= 0.2 -> fully optimized pockets but not realistic
# balanced 0.2 < 'x' <= 0.5  ->  balance between biochemical optimization and diversity
# higher 0.5 < 'x' <= 1 -> flatter prob. distribution, exploration-oriented
# essentially random: 1+
print(type(pocket_data))

<class 'dict'>


In [16]:
save_nano_pocket(pocket_data=pocket_data, filename="test_drug")

Pocket data for CC1=C2C(C(=C(C3=C(C(=O)C(C3(C1(C(CC2(C(C(C(C(=O)O)C(C(=O)NC(C(C5=CC=CC=C5)C(O)C6=CC=CC=C6)=O)O)C)C)OC(=O)C)(C)O)O)C)O)C)C)OC(=O)C)OC(=O)C saved:
Find nano pocket file in: <_io.TextIOWrapper name='/home/elliot/Desktop/LOCAL_PROJECTS/nano_maker/src/output_container/test_drug.nanopkt' mode='w' encoding='UTF-8'>


True

In [17]:
nanopkt_data = load_nano_pocket(filename="test_drug")
if nanopkt_data == pocket_data:
    print("Data Aligned")
# ensure the file saving / loading workflow is consistent in loading the data

Data Aligned


Now that we have generated a "nanopocket" for a given smiles and its data is saved, we can visualize the generated protein binding pocket with the following cell / module.

In [21]:
from nano_maker.pocketwatcher import PocketWatcher

visualizer = PocketWatcher(naanoeng_config=naanoeng_config, pocket_config=pocketwatcher_config)
visualizer.ingest_file(pocket_filepath="test_drug")
visualizer.visualize_pocket(identifier="charge_env")
# the identifier parameter allows users to distinguish the pocket's amino acids from each other either by plain cosmetic or by their biochemical properties. These are actually "summaries" of many features from the feature vectors engineered for NAAnoBot.

# purely cosmetic identifier settings
# [1] 'skeleton' -> just shows amino acids as white
# [2] 'color_code' -> color codes amino acids
# biochemical property identifier settings
# [3] 'charge_env' -> summary metric of AA's electrical environment, greater = polar
# [4] 'hydrophobicity' -> greater = hydrophobic, lower = hydrophilic
# [5] 'net_flex' -> composite metric of AA's flexibility, greater = more freedom, lower = restricted
# [6] 'net_steric' -> AA's size, weight, and exposed surface, greater = larger / more exposed

PocketWatcher also allows one to generate reports on the generated pockets for more quantitative analyses on the produced pockets.